# VIX Forecasting with LSTM (Multi-feature)

Predicts the daily open of the VIX volatility index using an LSTM trained on combined VIX (Open, High, Low, Close) and S&P 500 (Open, High, Low, Close) features.

The notebook is fully reproducible: data is pulled directly from Yahoo Finance via `yfinance`, so no manual data preparation is required. Training takes 5-10 minutes on a modern laptop CPU.


## Setup: install dependencies

Installs everything the notebook needs. Safe to run multiple times, pip will skip anything already installed.

If you prefer to manage your environment yourself, you can skip this cell and instead run `pip install -r requirements.txt` from a terminal.


In [ ]:
# Install the libraries this notebook depends on.
# The leading ! tells Jupyter to run the command in a shell, not as Python.
# -q (quiet) suppresses most pip output.
!pip install -q numpy pandas tensorflow scikit-learn yfinance matplotlib


## Parameters

All the tunable hyperparameters in one place so they are easy to adjust.


In [ ]:
# Sequence and model hyperparameters
n_steps   = 30       # number of past trading days the LSTM looks at
patience  = 30       # epochs without val_loss improvement before EarlyStopping fires
epoch     = 100      # max training epochs
sizetest  = 0.2      # fraction of the training period held out for validation
sizetrain = 0.8      # fraction used for training (must sum to 1 with sizetest)
sizebatch = 32       # batch size used during training
dropout   = 0.15     # dropout rate after each LSTM layer
learnrate = 0.0003   # learning rate for the Adam optimiser


## Imports

In [ ]:
# Numerical / dataframe stack
import numpy as np
import pandas as pd

# Deep learning (Keras lives inside TensorFlow)
import tensorflow as tf
from tensorflow import keras
from keras.models import Sequential
from keras.layers import LSTM, Dense, Dropout
from keras.callbacks import EarlyStopping

# Data preprocessing and train/test split
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split

# Data source: pulls historical market data from Yahoo Finance
import yfinance as yf

# Plotting
import matplotlib.pyplot as plt

# For saving plot outputs to disk
import os


## Data Download

Pulls historic VIX and S&P 500 data directly from Yahoo Finance via `yfinance`.

The training set covers a 10-year period; the evaluation set covers a separate later period so the model is never tested on data it was trained on.


In [ ]:
# Date ranges to pull from Yahoo Finance.
# Training data is the older 10-year period; evaluation is a separate, more recent year.
# Keeping them non-overlapping is what makes the evaluation honest.
train_start = '2014-01-01'
train_end   = '2024-01-01'
eval_start  = '2024-01-01'
eval_end    = '2025-01-01'

# ^VIX  -> CBOE Volatility Index
# ^GSPC -> S&P 500 Index
# auto_adjust=False keeps Open/High/Low/Close as raw values rather than adjusted.
vix_train = yf.download('^VIX',  start=train_start, end=train_end, progress=False, auto_adjust=False)
snp_train = yf.download('^GSPC', start=train_start, end=train_end, progress=False, auto_adjust=False)
vix_eval  = yf.download('^VIX',  start=eval_start,  end=eval_end,  progress=False, auto_adjust=False)
snp_eval  = yf.download('^GSPC', start=eval_start,  end=eval_end,  progress=False, auto_adjust=False)

print(f'VIX train: {len(vix_train)} rows, S&P train: {len(snp_train)} rows')
print(f'VIX eval:  {len(vix_eval)} rows, S&P eval:  {len(snp_eval)} rows')


## Feature Engineering

Combines VIX (Open, High, Low, Close) and S&P 500 (Open, High, Low, Close) into a single feature matrix.

The model uses 8 features per timestep (4 from VIX, 4 from S&P) but predicts only one target (VIX Open).


In [ ]:
def prepare_features(vix_df, snp_df):
    # Combine VIX (OHLC) and S&P (OHLC) into one feature matrix.
    # yfinance sometimes returns a multi-index column when given a single ticker;
    # flatten it so we can index by the simple column name.
    if isinstance(vix_df.columns, pd.MultiIndex):
        vix_df = vix_df.copy()
        vix_df.columns = [c[0] for c in vix_df.columns]
    if isinstance(snp_df.columns, pd.MultiIndex):
        snp_df = snp_df.copy()
        snp_df.columns = [c[0] for c in snp_df.columns]

    # Build the feature DataFrame with explicit, clearly-named columns.
    # Order matters: VIX Open is column 0, which is what we predict later.
    combined = pd.DataFrame({
        'vix_open':  vix_df['Open'].values,
        'vix_high':  vix_df['High'].values,
        'vix_low':   vix_df['Low'].values,
        'vix_close': vix_df['Close'].values,
        'snp_open':  snp_df['Open'].values,
        'snp_high':  snp_df['High'].values,
        'snp_low':   snp_df['Low'].values,
        'snp_close': snp_df['Close'].values,
    })
    # Drop any rows with missing data and reset the index.
    return combined.dropna().reset_index(drop=True)


train_data = prepare_features(vix_train, snp_train)
eval_data  = prepare_features(vix_eval, snp_eval)
print(f'Train data shape: {train_data.shape}')
print(f'Eval data shape:  {eval_data.shape}')
train_data.head()


## Train/Test Split and Scaling

**Methodology note:** the train/test split happens **before** scaling.

The MinMaxScaler is fitted only on the training portion, and the same fitted scaler is used to transform the test and evaluation data. Fitting the scaler on the full dataset before splitting would leak information about the test period's value range into training.


In [ ]:
# Convert the DataFrame to a plain numpy array for the rest of the pipeline.
train_array = train_data.values
eval_array  = eval_data.values

# Split FIRST. shuffle=False because this is time-series data we must preserve chronological order.
train, test = train_test_split(
    train_array,
    test_size=sizetest,
    train_size=sizetrain,
    shuffle=False,
)

# Fit the scaler ONLY on training data, then transform test and eval
# using the same fitted scaler. This prevents data leakage.
sc = MinMaxScaler(feature_range=(-1, 1))
train_scaled = sc.fit_transform(train)
test_scaled  = sc.transform(test)
eval_scaled  = sc.transform(eval_array)

print(f'Train scaled: {train_scaled.shape}, Test scaled: {test_scaled.shape}')
print(f'Eval scaled:  {eval_scaled.shape}')


## Sequence Creation

Converts the time series into sliding windows of length `n_steps`. The target at each window is the next-day VIX Open value (column 0).

Each sample is a window of 30 days, each day with 8 features. The label is the VIX Open on day 31.


In [ ]:
def split_sequences(sequences, n_steps, target_col=0):
    # Convert a 2D array (time x features) into sliding windows.
    # Returns X with shape (samples, n_steps, n_features) and y with shape (samples,).
    # The label is taken from target_col at the step immediately after the window.
    X, y = [], []
    for i in range(len(sequences) - n_steps):
        seq_x = sequences[i : i + n_steps, :]
        seq_y = sequences[i + n_steps, target_col]
        X.append(seq_x)
        y.append(seq_y)
    return np.array(X), np.array(y)


Xtrain, ytrain = split_sequences(train_scaled, n_steps)
Xtest,  ytest  = split_sequences(test_scaled,  n_steps)

n_features = Xtrain.shape[2]
print(f'Xtrain: {Xtrain.shape}, ytrain: {ytrain.shape}')
print(f'Xtest:  {Xtest.shape},  ytest:  {ytest.shape}')
print(f'n_features: {n_features}')


## Model

Stacked LSTM that takes 8 features per timestep and outputs a single value (next-day VIX Open).


In [ ]:
model = Sequential([
    # First LSTM with return_sequences=True so the next LSTM gets a full sequence.
    LSTM(units=128, input_shape=(n_steps, n_features), return_sequences=True),
    Dropout(dropout),

    # Second LSTM compresses the sequence into a single vector.
    LSTM(units=64),
    Dropout(dropout),

    # Dense head that maps the LSTM output down to a single prediction value.
    Dense(units=32, activation='relu'),
    Dense(units=16, activation='relu'),
    Dense(units=1),
])

# Adam optimiser with the learning rate from the parameters cell.
optimizer = keras.optimizers.Adam(learning_rate=learnrate)
model.compile(optimizer=optimizer, loss='mean_squared_error')

# EarlyStopping halts training when val_loss stops improving for `patience` epochs.
early_stop = EarlyStopping(monitor='val_loss', patience=patience)

model.summary()


## Training

Trains for up to `epoch` epochs, with EarlyStopping watching validation loss.


In [ ]:
history = model.fit(
    x=Xtrain,
    y=ytrain,
    validation_data=(Xtest, ytest),
    epochs=epoch,
    batch_size=sizebatch,
    callbacks=[early_stop],
    verbose=1,
)


## Loss Curves

Training and validation loss over epochs.


In [ ]:
# Pull both loss curves into one DataFrame so we can plot them together.
loss_df = pd.DataFrame({
    'train_loss': history.history['loss'],
    'val_loss':   history.history['val_loss'],
})
loss_df.plot(figsize=(10, 6))
plt.title('LSTM Training and Validation Loss', fontweight='bold')
plt.xlabel('Epoch', fontweight='bold')
plt.ylabel('Loss (MSE)', fontweight='bold')

# Save plot to results/ folder (creates it if missing).
os.makedirs('results', exist_ok=True)
plt.savefig('results/loss_curves.png', dpi=120, bbox_inches='tight')
plt.show()


## Predictions on the Evaluation Set

Generates rolling predictions over the held-out evaluation period. Plots predicted vs actual VIX Open (scaled).

The model has not seen any of this data during training, so this is a fair test of generalisation.


In [ ]:
def make_predictions(scaled_array, model, n_steps):
    # Slide a window across the input array and call the model for each window.
    # Returns a 1D array of predicted VIX Open values (one per window).
    preds = []
    for i in range(len(scaled_array) - n_steps):
        x_input = scaled_array[i : i + n_steps].reshape((1, n_steps, n_features))
        yhat = model.predict(x_input, verbose=0)
        preds.append(yhat[0, 0])
    return np.array(preds)


predict_eval = make_predictions(eval_scaled, model, n_steps)

# Actual values to compare against: skip the first n_steps because those are the input window
# for the first prediction (we have no prediction for them).
actual_vix_open = eval_scaled[n_steps:, 0]

print(f'Predictions: {len(predict_eval)}, Actual: {len(actual_vix_open)}')


In [ ]:
# Plot of actual vs predicted VIX Open on the held-out eval period.
plt.figure(figsize=(10, 6))
plt.plot(actual_vix_open, 'g-', label='Actual VIX Open (scaled)')
plt.plot(predict_eval,    'r-', label='Predicted VIX Open (scaled)')
plt.legend()
plt.title('VIX Open: Predicted vs Actual (Evaluation Period)')
plt.xlabel('Day')
plt.ylabel('Scaled VIX Open')
plt.savefig('results/eval_predictions.png', dpi=120, bbox_inches='tight')
plt.show()


## Zoomed Predictions Window

Plotting all evaluation days at once makes it hard to see day-by-day detail. This zoomed view lets you inspect a smaller window so you can judge whether the model is tracking turning points or just lagging.

Change `window_start` and `window_end` below to inspect different portions of the evaluation period.


In [ ]:
# Pick a window of days from the evaluation period to inspect closely.
window_start = 0
window_end   = 50

plt.figure(figsize=(10, 6))
plt.plot(actual_vix_open[window_start:window_end], 'g-', label='Actual VIX Open (scaled)', marker='o', markersize=4)
plt.plot(predict_eval[window_start:window_end],    'r-', label='Predicted VIX Open (scaled)', marker='x', markersize=4)
plt.legend()
plt.title(f'VIX Open: Predicted vs Actual (Days {window_start} to {window_end})')
plt.xlabel('Day within window')
plt.ylabel('Scaled VIX Open')
plt.savefig('results/eval_predictions_zoomed.png', dpi=120, bbox_inches='tight')
plt.show()


## Day-over-Day Change Comparison

Plots the day-over-day change of actual vs predicted. When both lines sit on the same side of zero on a given day, the model called direction correctly.


In [ ]:
# Compute day-over-day changes (today minus yesterday) for both series.
actual_change  = np.diff(actual_vix_open)
predict_change = np.diff(predict_eval)

# Plot a window so the per-day movements are visible.
a = 50
b = 150
plt.figure(figsize=(10, 6))
plt.plot(actual_change[a:b],  'g-', label='Actual day-over-day change',  marker='o', markersize=3)
plt.plot(predict_change[a:b], 'r-', label='Predicted day-over-day change', marker='x', markersize=3)
plt.axhline(0, color='gray', linestyle='--', linewidth=0.8)
plt.legend()
plt.title(f'Day-over-Day Change: Actual vs Predicted (Days {a} to {b})')
plt.xlabel('Day within window')
plt.ylabel('Change in scaled VIX Open')
plt.savefig('results/day_over_day_change.png', dpi=120, bbox_inches='tight')
plt.show()


## Directional Accuracy

MSE measures how close predictions are to actual values in level terms. For trading-style questions, directional accuracy (did the model correctly call whether tomorrow's VIX will be up or down vs today's) is often more useful.

A purely random model would score around 50%. Anything meaningfully above 50% suggests the model captures useful signal.


In [ ]:
def direction(series):
    # Return +1 for up, -1 for down, 0 for flat between consecutive points.
    return [
        1 if series[i+1] > series[i] else -1 if series[i+1] < series[i] else 0
        for i in range(len(series) - 1)
    ]


actual_dir = direction(actual_vix_open)
pred_dir   = direction(predict_eval)

# Count days where predicted direction agrees with actual direction.
correct = sum(a == p for a, p in zip(actual_dir, pred_dir))
total   = len(actual_dir)
accuracy = correct / total if total else 0.0

print(f'Directional accuracy: {correct}/{total} = {accuracy:.1%}')


## Baseline Comparisons

A directional accuracy number alone is hard to interpret. Comparing it against simple non-ML baselines makes the result concrete.

- **Random**: 50% by definition; the floor for any directional classifier.
- **Persistence direction** (also called the naive momentum forecast): predicts that tomorrow's direction will be the same as today's. This is the standard non-ML benchmark for direction forecasting.

An MSE-based naive level forecast (`tomorrow = today`) is also reported alongside the LSTM's MSE for level-prediction comparison.


In [ ]:
# Naive level forecast: predict tomorrow's value = today's actual value.
naive_level_preds   = actual_vix_open[:-1]
naive_level_actuals = actual_vix_open[1:]
naive_mse = float(np.mean((naive_level_actuals - naive_level_preds) ** 2))

# LSTM MSE on the same evaluation window
lstm_mse = float(np.mean((actual_vix_open - predict_eval) ** 2))

# Persistence direction baseline: predict tomorrow's direction = yesterday-to-today's direction.
persistence_pred_dir   = actual_dir[:-1]
persistence_actual_dir = actual_dir[1:]
persistence_correct = sum(a == p for a, p in zip(persistence_actual_dir, persistence_pred_dir))
persistence_total   = len(persistence_actual_dir)
persistence_accuracy = persistence_correct / persistence_total if persistence_total else 0.0

# Print comparison table
print('Baseline comparisons (evaluation period)')
print('=' * 65)
print(f"  {'Random (theoretical):':<35} 50.0% directional")
print(f"  {'Persistence direction:':<35} {persistence_accuracy:.1%} directional")
print(f"  {'Naive level (today=tomorrow):':<35} MSE = {naive_mse:.5f}")
print(f"  {'LSTM model:':<35} {accuracy:.1%} directional, MSE = {lstm_mse:.5f}")
print('=' * 65)
print()
print(f'LSTM beats random by:      {accuracy - 0.5:+.1%}')
print(f'LSTM beats persistence by: {accuracy - persistence_accuracy:+.1%}')
print(f'LSTM beats naive level by: {(naive_mse - lstm_mse) / naive_mse:+.1%} reduction in MSE')


## Save Model (optional)

Uncomment the line below if you want to save the trained model to disk.


In [ ]:
# Save the trained model in Keras's native format.
# Reload later with: model = keras.models.load_model('vix_model.keras')
# model.save('vix_model.keras')


## Notes / Limitations

- **Single training run**: results will vary between runs due to random weight initialisation. For a robust evaluation, average results over multiple seeds.
- **Single evaluation period**: the 2024 evaluation included the August carry-trade unwind. Different years may give different results. Multi-period rolling evaluation would strengthen the claim.
- **No transaction cost modelling**: directional accuracy is not the same as profit. Trading VIX derivatives carries spreads, slippage, and roll costs that would consume some or all of the apparent edge.
- **Level-based loss**: training on MSE optimises for value prediction, not for direction. A classification head trained with cross-entropy may produce sharper directional decisions.
- **Single-step prediction**: this model predicts one day ahead. Multi-step horizons would require additional logic.
- **Anchor bias in quiet regimes**: during sustained low-volatility periods, predictions sit higher than actuals because the model learned a mean-reversion bias from the training distribution. Predicting returns rather than levels would mitigate this.
